# Week 4 — NT Crime Lakehouse ingestion
This notebook loads the crime fact table and ABS SA2 geography into Delta tables for a Fabric semantic model and Direct Lake reporting.


In [ ]:
from pyspark.sql import functions as F

crime_path = 'Files/week4/nt_crime_statistics_june_2026.csv'
crime = (spark.read.option('header', True).option('inferSchema', True).csv(crime_path)
    .withColumnRenamed('As At', 'Snapshot Date')
    .withColumnRenamed('Month number', 'Month Number')
    .withColumnRenamed('Offence type ', 'Offence type')
    .withColumn('Month Start', F.make_date('Year', 'Month Number', F.lit(1)))
    .withColumn('Year Month', F.date_format('Month Start', 'yyyy-MM'))
    .withColumn('Statistical Area 2', F.trim(F.col('Statistical Area 2'))))
crime.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('nt_crime_fact')
display(crime.groupBy('Year Month').agg(F.sum('Number of offences').alias('Total Offences')).orderBy('Year Month'))


In [ ]:
geo_path = 'Files/week4/nt_sa2_centroids.csv'
geography = spark.read.option('header', True).option('inferSchema', True).csv(geo_path)
geography.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('nt_sa2_geography')
display(geography.select('SA2 Code', 'SA2 Name', 'Latitude', 'Longitude'))


In [ ]:
from notebookutils import mssparkutils
import json
raw = mssparkutils.fs.head('Files/week4/nt_sa2_2021_powerbi.geojson', 5000000)
features = json.loads(raw)['features']
boundary_rows = [(f['properties']['SA2 Code'], f['properties']['SA2 Name'], json.dumps(f['geometry'])) for f in features]
boundaries = spark.createDataFrame(boundary_rows, ['SA2 Code', 'SA2 Name', 'Geometry GeoJSON'])
boundaries.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('nt_sa2_boundaries')


## Direct Lake semantic model
Create a semantic model on the Lakehouse SQL analytics endpoint, relate `nt_crime_fact[Statistical Area 2]` to `nt_sa2_geography[SA2 Name]`, and choose Direct Lake storage mode. Reuse the DAX measures in `direct_lake_measures.dax`.
